<a href="https://colab.research.google.com/github/lakshitasingh/AQI-Geospatial-Analysis-System-Data-Analysis-Project-/blob/main/AQI_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install folium --quiet
print('Ready!')

from google.colab import files
import pandas as pd
import io

print('city_day.csv')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
df = df[['City', 'Date', 'PM2.5', 'AQI']].copy()
df.columns = ['city', 'date', 'pm25', 'aqi']
df['date'] = pd.to_datetime(df['date'])
df['aqi']  = pd.to_numeric(df['aqi'], errors='coerce')
df = df.dropna(subset=['aqi'])

print(f'\nLoaded successfully!')
print(f' {df["city"].nunique()} cities available')
print(f' Date range in file: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'\n Available cities:')
print(sorted(df['city'].unique().tolist()))

CITIES = 'ALL'

START_DATE = '2015-01-01'   # format: YYYY-MM-DD
END_DATE   = '2015-01-07'   # format: YYYY-MM-DD


if CITIES.strip().upper() == 'ALL':
    city_list = df['city'].unique().tolist()
else:
    city_list = [c.strip() for c in CITIES.split(',')]

print(f' Cities  : {city_list}')
print(f' From    : {START_DATE}  →  {END_DATE}')

import folium
from folium.plugins import HeatMap

# ── City coordinates ──
city_coords = {
    'Ahmedabad': (23.0225, 72.5714), 'Aizawl': (23.7271, 92.7176),
    'Amaravati': (20.9374, 77.7796), 'Amritsar': (31.6340, 74.8723),
    'Bengaluru': (12.9716, 77.5946), 'Bhopal': (23.2599, 77.4126),
    'Brajrajnagar': (21.8218, 83.918), 'Chandigarh': (30.7333, 76.7794),
    'Chennai': (13.0827, 80.2707),   'Coimbatore': (11.0168, 76.9558),
    'Delhi': (28.6139, 77.2090),     'Ernakulam': (9.9816, 76.2999),
    'Gurugram': (28.4595, 77.0266),  'Guwahati': (26.1445, 91.7362),
    'Hyderabad': (17.3850, 78.4867), 'Jaipur': (26.9124, 75.7873),
    'Jorapokhar': (23.6800, 86.4200),'Kochi': (9.9312, 76.2673),
    'Kolkata': (22.5726, 88.3639),   'Lucknow': (26.8467, 80.9462),
    'Mumbai': (19.0760, 72.8777),    'Patna': (25.5941, 85.1376),
    'Shillong': (25.5788, 91.8933),  'Talcher': (20.9500, 85.2333),
    'Thiruvananthapuram': (8.5241, 76.9366),
    'Visakhapatnam': (17.6868, 83.2185),
}

def get_aqi_category(aqi):
    if aqi <= 50:    return 'Good'
    elif aqi <= 100: return 'Satisfactory'
    elif aqi <= 200: return 'Moderate'
    elif aqi <= 300: return 'Poor'
    elif aqi <= 400: return 'Very Poor'
    else:            return 'Severe'

def get_color(aqi):
    if aqi <= 50:    return '#00cc44'
    elif aqi <= 100: return '#aadd00'
    elif aqi <= 200: return '#ffaa00'
    elif aqi <= 300: return '#ff4400'
    elif aqi <= 400: return '#cc0000'
    else:            return '#660000'

# Reading widget values
selected = list(city_selector.value)
start    = str(start_picker.value)
end      = str(end_picker.value)
city_list = df['city'].unique().tolist() if 'ALL' in selected else selected

filtered = df[
    df['city'].isin(city_list) &
    (df['date'] >= start) &
    (df['date'] <= end)
]

if filtered.empty:
    print('No data found. Try a different city or date range.')
else:
    summary = filtered.groupby('city').agg(
        avg_aqi  = ('aqi',  'mean'),
        max_aqi  = ('aqi',  'max'),
        min_aqi  = ('aqi',  'min'),
        avg_pm25 = ('pm25', 'mean')
    ).reset_index()
    summary['avg_aqi']  = summary['avg_aqi'].round(1)
    summary['avg_pm25'] = summary['avg_pm25'].round(1)
    summary['category'] = summary['avg_aqi'].apply(get_aqi_category)
    summary['lat'] = summary['city'].map(lambda c: city_coords.get(c, (None,None))[0])
    summary['lon'] = summary['city'].map(lambda c: city_coords.get(c, (None,None))[1])
    summary = summary.dropna(subset=['lat', 'lon'])

    india_map = folium.Map(
        location   = [22.5, 82.0],
        zoom_start = 5,
        min_zoom   = 5,
        max_zoom   = 10,
        tiles      = 'CartoDB dark_matter',
        max_bounds = True
    )
    HeatMap(
        [[r.lat, r.lon, r.avg_aqi] for r in summary.itertuples()],
        radius=50, blur=35, min_opacity=0.5,
        gradient={'0.0':'#0000ff','0.25':'#00cc44','0.5':'#ffdd00','0.75':'#ff6600','1.0':'#cc0000'}
    ).add_to(india_map)

    for r in summary.itertuples():
        color = get_color(r.avg_aqi)
        folium.CircleMarker(
            location     = [r.lat, r.lon],
            radius       = 9,
            color        = 'white',
            weight       = 1,
            fill         = True,
            fill_color   = color,
            fill_opacity = 0.9,
            tooltip      = f'{r.city} | AQI: {r.avg_aqi} | {r.category}',
            popup        = folium.Popup(f"""
                <div style='font-family:Arial;font-size:13px;width:200px;line-height:1.8'>
                <b style='font-size:15px'>🏙️ {r.city}</b><hr style='margin:4px 0'>
                <b>Avg AQI:</b> {r.avg_aqi}<br>
                <b>Level:</b> <span style='color:{color};font-weight:bold'>{r.category}</span><br>
                <b>Max AQI:</b> {r.max_aqi}<br>
                <b>Avg PM2.5:</b> {r.avg_pm25} µg/m³
                </div>""", max_width=220)
        ).add_to(india_map)

    # Legend
    india_map.get_root().html.add_child(folium.Element("""
    <div style='position:fixed;bottom:40px;left:40px;background:rgba(0,0,0,0.82);
         color:white;padding:14px 18px;border-radius:10px;font-family:Arial;
         font-size:12px;z-index:9999;line-height:2.0;border:1px solid #555'>
        <b>AQI Color Scale</b><br>
        <span style='color:#00cc44'>●</span> Good (0–50)<br>
        <span style='color:#aadd00'>●</span> Satisfactory (51–100)<br>
        <span style='color:#ffaa00'>●</span> Moderate (101–200)<br>
        <span style='color:#ff4400'>●</span> Poor (201–300)<br>
        <span style='color:#cc0000'>●</span> Very Poor (301–400)<br>
        <span style='color:#660000'>●</span> Severe (400+)
    </div>
    """))

    india_map.save('india_aqi_heatmap.html')
    print(f'Map generated for {summary.shape[0]} cities | {start} → {end}')
    display(india_map)